### RAG pipeline - data ingestion to vector db

In [3]:
# import required libraries
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv('HUGGING_FACE_API_KEY')

/tmp/ipykernel_231167/852626585.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, DirectoryLoader


In [2]:
# Read all the pdf inside diractory
def load_all_pdf(path_to_pdfs):
    """Load and process all the pdfs"""
    all_documents = []
    path_dir = Path(path_to_pdfs)

    # load all pdf as list
    files = list(path_dir.glob('**/*.pdf'))

    if not files:
        return None
    print(f"Found {len(files)} pdfs.")
    
    for file in files:
        print(f"prpcessing {file}")
        try:
            loader = PyMuPDFLoader(str(file))
            documents = loader.load()
            print(f'found {len(documents)} pages\n')

            for doc in documents:
                doc.metadata['source_file'] = file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
        except Exception as e:
            print(f"Error: {e}")
    
    return all_documents


# less control over the files. Instead return all documents at once
def load_all_pdf2(dir):
    loader = DirectoryLoader(
        dir,
        glob='**/*.pdf',
        loader_cls=PyMuPDFLoader
    )
    all_documents = loader.load()

    for doc in all_documents:
        doc.metadata['source_file'] = Path(doc.metadata['file_path']).name
        doc.metadata['file_type'] = 'pdf'
    
    return all_documents

documents = load_all_pdf('../data/pdf/AstroPhysics/')

Found 4 pdfs.
prpcessing ../data/pdf/AstroPhysics/ImpactofPerfectFluidDarkMatter.pdf
found 33 pages

prpcessing ../data/pdf/AstroPhysics/PropagatingInstabilityforWaveDarkMatter.pdf
found 33 pages

prpcessing ../data/pdf/AstroPhysics/ParticlePhysicsandModern.pdf
found 24 pages

prpcessing ../data/pdf/AstroPhysics/Core-HaloRelationofQuantumWaveDarkMatter.pdf
found 6 pages



In [3]:
# Split into chunks 
def split_text(docs, chunk_size = 1000, chunk_overlap = 200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=['\n\n', '\n', '. ', '! ', '? ', ' ', '']
    )
    chunk = splitter.split_documents(documents=docs)
    print(f'Splitted {len(docs)} documents into {len(chunk)} chunk')

    if chunk:
        print(f'{chunk[0].metadata}')
        print(f'{chunk[0].page_content[:200]}')

    return chunk

chunks = split_text(documents)

Splitted 96 documents into 382 chunk
{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'source': '../data/pdf/AstroPhysics/ImpactofPerfectFluidDarkMatter.pdf', 'file_path': '../data/pdf/AstroPhysics/ImpactofPerfectFluidDarkMatter.pdf', 'total_pages': 33, 'format': 'PDF 1.7', 'title': 'Impact of Perfect Fluid Dark Matter on the Appearance of Rotating Black Hole', 'author': 'Huang Yu-Xiang; Guo Sen; Liang En-Wei; Lin Kai', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'ImpactofPerfectFluidDarkMatter.pdf', 'file_type': 'pdf'}
Impact of Perfect Fluid Dark Matter on the
Appearance of Rotating Black Hole
Yu-Xiang Huanga, Sen Guob,∗, En-Wei Lianga,∗, Kai Linc
aGuangxi Key Laboratory for Relativistic Astrophysics, School of Phy


In [4]:
models = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'BAAI/bge-base-en-v1.5',    
    "BAAI/bge-small-en-v1.5",
    "intfloat/e5-base-v2"
]

In [5]:
## Embedding and vector store
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_chroma import Chroma


hf_embeddings = HuggingFaceEndpointEmbeddings(  
    model="sentence-transformers/all-mpnet-base-v2",  
    task="feature-extraction",  
    huggingfacehub_api_token=HF_TOKEN,  
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=hf_embeddings,
    collection_name='chromaDb',
    persist_directory='../chroma_db'
)

print("Vector store created successfully!")

/home/desktop-potato/Desktop/ai/prj/rag-app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector store created successfully!


In [6]:
from typing import List

class RAGRetriever:
    """Handles query based retrival from vector-store"""

    def __init__(self, vector_store = vector_store, embeddings = hf_embeddings):
        """
        Initialize Retriever
        ARG:
            vector_store: vector store containing the embeddings
            embeddings: embedding function to handle query embedding
        """

        self.vector_store = vector_store
        self.embeddings = embeddings
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[dict[str, any]]:
        """
        Retrieve relevent document for a querry 

        arg:
            query: The search query 
            top_k: Number of top best result to return 
            score_threshold: Minimum similarity score
        Returns:
            List of relevent document and metadata
        """

        print(f'Retrieving relevent data for querry "{query}"')
        print(f'Top_k: {top_k}, score_threshold: {score_threshold}')

        # Generating query embeddings
        query_embedding = self.embeddings.embed_query(query)

        # Search in vector db
        try:
            results = vector_store._collection.query(
                query_embeddings=query_embedding,
                n_results=top_k
            )

            # process result
            retrived_docs = []

            if results['documents'] and results['documents'][0]:
                doc_ids = results['ids'][0]
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(doc_ids, documents, metadatas, distances)):
                    # Convert distance into similarity score
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrived_docs.append({
                            'doc_id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f'Retrive {len(retrived_docs)} documents after filtering.')
            else:
                print('No document found')
            
            return retrived_docs
                

        except Exception as e:
            print(f'Exception occured with {e}')

retriver = RAGRetriever()

In [7]:
#  An example how to use the retriver
result = retriver.retrieve("Dark matter")

Retrieving relevent data for querry "Dark matter"
Top_k: 5, score_threshold: 0.0
Retrive 5 documents after filtering.


In [4]:
groq_model = "llama-3.1-8b-instant"
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [6]:
from langchain_groq import ChatGroq

# Initialize the groq llm
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=groq_model,
    temperature=0.1,
    max_token = 1024
)

/home/desktop-potato/Desktop/ai/prj/rag-app/.venv/lib/python3.12/site-packages/pydantic/main.py:263: UserWarning: WARNING! max_token is not default parameter.
                    max_token was transferred to model_kwargs.
                    Please confirm that max_token is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
